# 08 · P2 平台化 — agent.yaml 部署契约 + Model Router + Deployment Profiles

`config/agent.yaml`(nemo-agents-spec-v1) 自洽校验; Model Mesh 角色→local/NIM/mock 路由; Profile A/B/C/D。

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from bootstrap import build_controller, status
from services.config import load_settings, load_policy
print('trunk root:', ROOT)

In [ ]:
from services.agent_spec import load_agent_spec, validate
spec = load_agent_spec(ROOT)
v = validate(spec)
print('agent.yaml valid=', v['valid'], '| tools=', v['tool_count'], '| roles=', v['roles'], '| profiles=', v['profiles'])
print('errors=', v['errors'], 'warnings=', v['warnings'])
assert v['valid']

In [ ]:
from services.model_router import ModelRouter
from services.config import load_settings
from adapters.timo_adapter import TimoAdapter
s = load_settings(ROOT); t = TimoAdapter(s)
mr = ModelRouter(s, timo=t)
st = mr.status()
print('backend=', st['backend'], '| online_roles=', st['online_roles'])
for role, r in st['routes'].items():
    print('  %-14s -> %-14s online=%-5s %s' % (role, r['backend'], r['online'], r.get('note','')[:40]))

In [ ]:
import yaml
prof = yaml.safe_load((ROOT/'deploy'/'profiles.yaml').read_text(encoding='utf-8'))
print('default profile:', prof['default'])
for k in ['A','B','C','D']:
    p = prof['profiles'][k]
    print('  Profile %s: %-26s engine=%s' % (k, p['name'], (p.get('engine') or {}).get('mode')))

## 部署产物清单 (deploy/)

In [ ]:
for f in ['Dockerfile','docker-compose.yml','k8s.yaml','profiles.yaml','hpa.yaml','grafana-dashboard.json']:
    p = ROOT/'deploy'/f
    print(('OK ' if p.exists() else 'MISSING '), f, p.stat().st_size if p.exists() else 0, 'bytes')